# Ноутбук 2. Feature Engineering и обучение модели

## Цели этапа

- Загрузить и подготовить данные
- Создать все признаки на основе выводов EDA
- Обучить финальную модель (LGBMRanker)
- Собрать ансамбль из трёх моделей с разными seed
- Сформировать сабмит для отправки



## 1. Импорт библиотек 

In [23]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Загрузка данных

In [24]:
train = pd.read_parquet('../data/train_dataset_small.pq')
test = pd.read_parquet('../data/test_dataset_small.pq')

print(f"train: {train.shape}")
print(f"test: {test.shape}")

train: (845110, 23)
test: (936883, 22)


## 2.1 Клиентские признаки (features_small.pq)

В ходе EDA был проведён анализ дополнительных клиентских признаков из файла features_small.pq.

- Уникальных app_id в train: 37 817
- Уникальных app_id в features: 76 435
- Пересечение train ∩ features: 37 817 (100%)

При объединении на уровне строк доля пропусков в клиентских признаках составила **82%**.

Поэтому использование features_small.pq в модели нецелесообразно — 82% пропусков 
создают шум. В финальный пайплайн эти признаки не включены.

## 2.2 Конвертация типов данных

Некоторые числовые колонки (`rate`, `eva`, `eva_perc`, `ncl`) хранятся как `Decimal` — 
это мешает работе numpy и LightGBM. Приводим их к `float`.

In [25]:
from decimal import Decimal
for df in [train, test]:
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                s = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
                if isinstance(s, Decimal):
                    df[col] = df[col].astype(float)
            except: pass

## 3. Очистка данных

В ходе EDA были обнаружены дубликаты по паре (app_id, offer_id). Одна и та же пара «клиент — кредитное предложение» встречается в данных дважды, причём в некоторых случаях с разным значением таргета (is_deal). Такие дубликаты вносят шум в обучение.

**Принятое решение:** удалить дубликаты, оставив первую запись. Предварительная сортировка по is_deal гарантирует, что если среди дубликатов есть сделка, она останется.

In [26]:
train = train.sort_values('is_deal', ascending=False)
train = train.drop_duplicates(subset=['app_id', 'offer_id'], keep='first')

print(f"train после удаления дубликатов {train.shape}")

train после удаления дубликатов (844245, 23)


**Результат:** из train удалено 865 строк (1730 дублирующихся строк превратились в 865 уникальных пар). Размер train теперь: 844 245 строк.

## 4. Кодирование категориальных признаков

Модели градиентного бустинга (LightGBM, XGBoost, CatBoost) работают с числовыми данными. Категориальные признаки необходимо преобразовать в числовой формат.

**Кодируемые признаки и их значения:**

| Исходный признак | Значения | Числовой код |
|-----------------|----------|--------------|
| channel | Internet, Front | 0, 1 |
| verif_need | N, Y | 0, 1 |
| need_2ndfl | N, D, Y | 0, 1, 2 |
| offer_type | RA, PL | 0, 1 |
| risk_level_map | VLR, MR, LR, HR | 0, 1, 2, 3 |
| verif_compl | N, Y | 0, 1 |

In [27]:
for df in [train, test]:
    df['ch'] = df['channel'].map({'Internet': 0, 'Front': 1})
    df['vn'] = df['verif_need'].map({'N': 0, 'Y': 1})
    df['nd'] = df['need_2ndfl'].map({'N': 0, 'D': 1, 'Y': 2})
    df['ot'] = df['offer_type'].map({'RA': 0, 'PL': 1})
    df['rl'] = df['risk_level_map'].map({'VLR': 0, 'MR': 1, 'LR': 2, 'HR': 3})
    df['vc'] = df['verif_compl'].map({'N': 0, 'Y': 1})


**Результат:** создано 6 новых числовых колонок (ch, vn, nd, ot, rl, vc). Исходные категориальные колонки (channel, verif_need, need_2ndfl, offer_type, risk_level_map, verif_compl) сохраняются, но в модель не включаются.

## 5. Matching features (признаки согласования с запросом клиента)

Создаём признаки, показывающие, насколько предложение (limit, term) соответствует тому, что запросил клиент. Если предложение сильно отличается от запроса, клиент с меньшей вероятностью его примет.

| Признак | Формула | Смысл |
|---------|---------|-------|
| `da` (diff_amount) | limit - req_loan_amount | На сколько предложенная сумма отличается от запрошенной (может быть отрицательной) |
| `dt` (diff_term) | term - req_term | На сколько предложенный срок отличается от запрошенного |
| `ada` (abs_diff_amount) | \|da\| | Модуль разницы по сумме (важно, насколько далеко, а не в какую сторону) |
| `adt` (abs_diff_term) | \|dt\| | Модуль разницы по сроку |
| `ra` (ratio_amount) | limit / req_loan_amount | Во сколько раз предложенная сумма больше запрошенной (защита от деления на 0) |
| `rt` (ratio_term) | term / req_term | Во сколько раз предложенный срок больше запрошенного (защита от деления на 0) |

При вычислении отношений добавлена защита от деления на ноль (если req_loan_amount или req_term равны 0, значение заменяется на 1).

In [28]:
for df in [train, test]:
    df['da'] = df['limit'] - df['req_loan_amount']
    df['dt'] = df['term'] - df['req_term']
    df['ada'] = np.abs(df['da'])
    df['adt'] = np.abs(df['dt'])
    df['ra'] = (df['limit'] / df['req_loan_amount'].replace(0, np.nan)).fillna(1)
    df['rt'] = (df['term'] / df['req_term'].replace(0, np.nan)).fillna(1)


## 6. Признаки из basket_name

Извлекаем из строкового поля basket_name бинарные флаги для ключевых бизнес-правил и подсчитываем общее количество правил в оффере.

| Признак | Описание |
|---------|----------|
| hdp | наличие правила desiredparams |
| hma | наличие правила minannuity |
| hml | наличие правила maxlimit |
| hnv | наличие правила noverifdocmaxlimit |
| brc | количество правил (через запятую) |

**Зачем это нужно:** в EDA было установлено, что в сделках чаще встречаются комбинации с maxlimit и minannuity, а в отказах — noverifdocmaxlimit. Эти признаки помогут модели улавливать такие паттерны.

In [29]:
for df in [train, test]:
    df['hdp'] = df['basket_name'].str.contains('desiredparams', na=False).astype(int)
    df['hma'] = df['basket_name'].str.contains('minannuity', na=False).astype(int)
    df['hml'] = df['basket_name'].str.contains('maxlimit', na=False).astype(int)
    df['hnv'] = df['basket_name'].str.contains('noverifdocmaxlimit', na=False).astype(int)
    df['brc'] = df['basket_name'].str.split(',').str.len().fillna(0).astype(int)



## 7. Новые комбинированные признаки

Создаём производные признаки, которые комбинируют информацию из исходных колонок.

| Признак | Формула | Описание |
|---------|---------|----------|
| rtn | rate / ncl | Доходность на единицу риска |
| ltt | limit / term | Средний ежемесячный платёж |
| rtr | req_loan_amount / limit | Отношение запрошенной суммы к предложенной |
| bs | vn + nd | Индекс бюрократии (сумма требований к документам) |
| ttr | term * rate / 100 | Произведение срока и ставки (характеризует общую стоимость кредита) |

**Обоснование:**

- **ltt (средний ежемесячный платёж):** клиенты часто оценивают кредит не по общей сумме, а по тому, сколько нужно платить каждый месяц. Низкий платёж повышает вероятность сделки.
- **rtn (доходность на единицу риска):** высокое значение означает, что кредит выгоден банку, но клиент может воспринимать рискованные кредиты как более доступные.
- **bs (бюрократия):** в EDA мы видели, что при необходимости подтверждения дохода (need_2ndfl=Y) конверсия падает в 18 раз. Чем выше bs, тем сложнее получить кредит.
- **ttr (общая стоимость):** чем выше, тем дороже кредит. Клиенты с большей вероятностью откажутся от дорогого предложения.
- **rtr (одобренная доля запроса):** если клиенту одобрили значительно меньше, чем он просил, он может отказаться.

In [30]:
for df in [train, test]:
    df['rtn'] = df['rate'] / df['ncl'].replace(0, 0.01)
    df['ltt'] = df['limit'] / df['term'].replace(0, 1)
    df['rtr'] = (df['req_loan_amount'] / df['limit'].replace(0, np.nan)).fillna(1)
    df['bs'] = df['vn'] + df['nd']
    df['ttr'] = df['term'] * df['rate'] / 100

## 8. Формирование финального набора признаков

На основе EDA и созданных признаков формируем итоговый список колонок для моделирования.

**Полный список (32 признака):**

| Группа | Признаки |
|--------|----------|
| Исходные (после кодирования) | variant_no, ot, rate, term, limit, rl, eva, eva_perc, ncl, vn, nd, vc, ch, pil1mtrx_offer, req_loan_amount, req_term |
| Matching features | da, dt, ra, rt, ada, adt |
| Basket name | hdp, hma, hml, hnv, brc |
| Новые | rtn, ltt, rtr, bs, ttr |

Итого: 32 признака.

In [31]:
FEATURES = [
    'variant_no', 'ot', 'rate', 'term', 'limit', 'rl', 'eva', 'eva_perc', 'ncl',
    'vn', 'nd', 'vc', 'ch', 'pil1mtrx_offer', 'req_loan_amount', 'req_term',
    'da', 'dt', 'ra', 'rt', 'ada', 'adt',
    'hdp', 'hma', 'hml', 'hnv', 'brc',
    'rtn', 'ltt', 'rtr', 'bs', 'ttr'
]

print(f"Всего признаков {len(FEATURES)}")


Всего признаков 32


## 9. Подготовка данных для LGBMRanker

LGBMRanker требует, чтобы данные были отсортированы по идентификатору группы (app_id). Также необходимо передать массив размеров групп.

In [32]:
X = train[FEATURES].copy()
y = train['is_deal'].copy()
groups = train['app_id'].copy()

sort_idx = groups.argsort()
X_sorted = X.iloc[sort_idx].reset_index(drop=True)
y_sorted = y.iloc[sort_idx].reset_index(drop=True)
groups_sorted = groups.iloc[sort_idx].reset_index(drop=True)

group_sizes = groups_sorted.value_counts(sort=False).reindex(
    groups_sorted.unique(), fill_value=0
).values

X_test = test[FEATURES].copy()

print(f"X_sorted shape: {X_sorted.shape}")
print(f"Количество групп (клиентов): {len(group_sizes):,}")
print(f"Сумма размеров групп: {group_sizes.sum():,}")

X_sorted shape: (844245, 32)
Количество групп (клиентов): 37,817
Сумма размеров групп: 844,245


## 10. Выбор модели и обоснование решений

### 10.1 Почему LGBMRanker

Задача — ранжирование кредитных предложений внутри одной заявки (app_id). Стандартные классификаторы (LGBMClassifier, XGBClassifier) оптимизируют accuracy или logloss, но не учитывают порядок предложений внутри группы.


**LGBMRanker** с функцией потерь `lambdarank` оптимизирует непосредственно метрику ранжирования NDCG@5, что соответствует целевой метрике соревнования.

### 10.2 Почему LGBMRanker

| Модель | Результат (NDCG@5) | Проблема |
|--------|--------------------|----------|
| LGBMRanker | 91.10 | Лучший результат |
| XGBRanker | 90.77 | Чуть хуже, дольше обучается |
| CatBoostRanker | 90.77 | Сходный с XGBoost, но медленнее |

Эмпирически установлено, что CatBoost и XGBoost не улучшают результат 
по сравнению с одиночным LGBMRanker. Ансамбль из трёх копий LGBMRanker 
с разными seed оказался эффективнее, чем комбинация разных моделей.

### 10.3 Почему ансамбль из трёх моделей

Ансамблирование снижает дисперсию предсказаний и повышает стабильность. Одна модель может переобучиться под конкретный seed или случайное разбиение. Комбинация трёх моделей с разными seed даёт более стабильный результат.

### 10.4 Веса ансамбля

Веса подобраны эмпирически на валидационной выборке:

- seed 42 — вес 0.5 (наилучшее качество)
- seed 123 — вес 0.3 (среднее качество)
- seed 456 — вес 0.2 (чуть хуже)

Взвешенное усреднение позволяет снизить влияние менее удачных моделей.

### 10.5 Выбор seed

Seed 42, 123, 456 выбраны произвольно. Разные seed влияют на начальную 
инициализацию деревьев, порядок данных в subsample и colsample_bytree, 
что обеспечивает разнообразие моделей в ансамбле.


## 11. Обучение ансамбля и формирование сабмита

In [33]:
# Параметры ансамбля
seeds = [42, 123, 456]
weights = [0.5, 0.3, 0.2]
predictions = []

for seed in seeds:
    model = lgb.LGBMRanker(
        objective='lambdarank',
        metric='ndcg',
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=32,
        min_child_samples=100,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=0.1,
        reg_alpha=0.1,
        random_state=seed,
        verbose=-1,
        n_jobs=-1
    )
    
    model.fit(X_sorted, y_sorted, group=group_sizes)
    pred = model.predict(X_test)
    
    # MinMax нормализация в [0,1]
    pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)
    predictions.append(pred)
    
    print(f"Модель с seed={seed} обучена, среднее предсказание: {pred.mean():.4f}")

# Взвешенное усреднение
ensemble = (weights[0] * predictions[0] + 
            weights[1] * predictions[1] + 
            weights[2] * predictions[2])


print(f"Среднее : {ensemble.mean():.4f}")
print(f"Мин: {ensemble.min():.4f}, Макс: {ensemble.max():.4f}")


Модель с seed=42 обучена, среднее предсказание: 0.3844
Модель с seed=123 обучена, среднее предсказание: 0.3808
Модель с seed=456 обучена, среднее предсказание: 0.3934
Среднее : 0.3851
Мин: 0.0090, Макс: 0.9906


In [34]:
import os

os.makedirs('../submissions', exist_ok=True)
submission = pd.DataFrame({
    'request_id': test['request_id'].astype(int),
    'variant_no': test['variant_no'].astype(int),
    'score': ensemble
})

submission.to_csv('../submissions/answers.csv', sep=';', index=False)
